# NetCDF File Reader

This notebook reads NetCDF (.nc) files and displays the first N values from each variable/column.

## Features:
- Load and inspect NetCDF file structure
- Display dimensions, variables, and attributes
- Print first N values from each variable
- **Special handling for Date columns**: Automatically detects and parses date strings stored as character arrays
- Configurable number of values to display
- Support for multi-dimensional arrays
- Comprehensive statistics (similar to pandas describe())

## Requirements:
- netCDF4
- numpy
- xarray (optional, for easier handling)
- datetime (for date parsing)

In [1]:
# Install required packages if not already installed
# Uncomment the lines below if needed

# !pip install netCDF4 numpy xarray

import netCDF4 as nc
import numpy as np
import os
from pathlib import Path

print("✓ Required packages imported successfully")
print(f"  netCDF4 version: {nc.__version__}")
print(f"  numpy version: {np.__version__}")


# /media/sentoki/kishodai/SSM-irrigation-data/NDWI NDVI data/NDVI_NDWI_combined.nc


# rsync -havzP /media/sentoki/kishodai/SSM-irrigation-data/NDWI_NDVI_data/NDVI_NDWI_combined.nc ishan@172.18.16.22:/master_storage6/ishan/my_data/SSM-irrigation-data/


✓ Required packages imported successfully
  netCDF4 version: 1.7.2
  numpy version: 2.1.0


In [2]:
# Optional: Import xarray for easier NetCDF handling
try:
    import xarray as xr
    print(f"✓ xarray available (version: {xr.__version__})")
    XARRAY_AVAILABLE = True
except ImportError:
    print("⚠ xarray not available. Using netCDF4 only.")
    print("  Install with: pip install xarray")
    XARRAY_AVAILABLE = False

✓ xarray available (version: 2025.1.1)


## Configuration

Set the file path and number of values to display:

In [3]:
# CONFIGURATION
# =============================================================================

# Path to your NetCDF file
nc_file_path =  r"G:\SM2RAIN-irrigation_Final\GPM_IMERG_ludhiana_final_run.nc" #r"G:\sm2rain-irrigation\GPM_IMERG_Daily_ROI_main.nc" #r'G:\agERA5\Temperature-Air-2m_Max-24h_C3S-glob-agric_AgERA5_20000101_final-v2.0.0.area-subset.31.76.30.75.nc'  # Change this to your file path

# Number of values to display from each variable
N = 10  # Change this to display more or fewer values

# =============================================================================

# Check if file exists
if os.path.exists(nc_file_path):
    print(f"✓ File found: {nc_file_path}")
    file_size = os.path.getsize(nc_file_path) / (1024 * 1024)  # Size in MB
    print(f"  File size: {file_size:.2f} MB")
else:
    print(f"✗ File not found: {nc_file_path}")
    print(f"  Please update the 'nc_file_path' variable with the correct path")

✓ File found: G:\SM2RAIN-irrigation_Final\GPM_IMERG_ludhiana_final_run.nc
  File size: 2.29 MB


## Method 1: Using netCDF4 Library

This method provides detailed access to NetCDF file structure.

In [4]:
# Open the NetCDF file
print("="*70)
print("OPENING NETCDF FILE")
print("="*70)

try:
    dataset = nc.Dataset(nc_file_path, 'r')
    print(f"✓ Successfully opened: {nc_file_path}")
    print()
    
    # Display global attributes
    print("📋 GLOBAL ATTRIBUTES:")
    print("-" * 70)
    if dataset.ncattrs():
        for attr in dataset.ncattrs():
            print(f"  {attr}: {dataset.getncattr(attr)}")
    else:
        print("  No global attributes found")
    print()
    
    # Display dimensions
    print("📏 DIMENSIONS:")
    print("-" * 70)
    for dim_name, dim in dataset.dimensions.items():
        print(f"  {dim_name}: {len(dim)} {'(unlimited)' if dim.isunlimited() else ''}")
    print()
    
    # Display variables
    print("📊 VARIABLES:")
    print("-" * 70)
    for var_name, var in dataset.variables.items():
        print(f"  {var_name}:")
        print(f"    Shape: {var.shape}")
        print(f"    Dimensions: {var.dimensions}")
        print(f"    Data type: {var.dtype}")
        if var.ncattrs():
            print(f"    Attributes:")
            for attr in var.ncattrs():
                attr_value = var.getncattr(attr)
                # Truncate long attributes
                if isinstance(attr_value, str) and len(attr_value) > 60:
                    attr_value = attr_value[:60] + "..."
                print(f"      {attr}: {attr_value}")
        print()
    
except Exception as e:
    print(f"✗ Error opening file: {str(e)}")
    dataset = None

OPENING NETCDF FILE
✓ Successfully opened: G:\SM2RAIN-irrigation_Final\GPM_IMERG_ludhiana_final_run.nc

📋 GLOBAL ATTRIBUTES:
----------------------------------------------------------------------
  title: GPM IMERG Daily Precipitation - ROI Filtered
  institution: NASA Goddard Space Flight Center
  source: GPM_3IMERGDF v07
  history: Created on 2026-02-07 01:39:42
  references: https://doi.org/10.5067/GPM/IMERGDF/DAY/07
  comment: Filtered for ROI: lon(74.95-76.05), lat(29.95-31.05)
  Conventions: CF-1.6

📏 DIMENSIONS:
----------------------------------------------------------------------
  time: 4018 (unlimited)
  date_strlen: 10 
  lat: 12 
  lon: 12 

📊 VARIABLES:
----------------------------------------------------------------------
  date:
    Shape: (4018, 10)
    Dimensions: ('time', 'date_strlen')
    Data type: |S1
    Attributes:
      units: DD-MM-YYYY
      standard_name: date
      long_name: Date
      description: Calendar date in DD-MM-YYYY format

  lat:
    Shape: (12

In [5]:
# Display first N values from each variable with comprehensive statistics
if dataset is not None:
    print("="*70)
    print(f"VARIABLE STATISTICS & FIRST {N} VALUES")
    print("="*70)
    print()
    
    for var_name, var in dataset.variables.items():
        print(f"📌 {var_name}")
        print(f"   Shape: {var.shape}, Dimensions: {var.dimensions}")
        
        try:
            # Get the data
            data = var[:]
            
            # Check if this is a Date/time variable stored as strings
            is_date_string = False
            date_values = []
            if var_name.lower() in ['date', 'datetime'] and data.dtype.kind in ['S', 'U', 'O']:
                try:
                    # Handle character array dates (e.g., shape (n, strlen))
                    if len(data.shape) == 2:
                        # Concatenate characters to form date strings
                        from datetime import datetime
                        for i in range(min(N, data.shape[0])):
                            date_str = ''.join([char.decode('utf-8') if isinstance(char, bytes) else str(char) 
                                              for char in data[i] if char not in [b'', '', b'--', '--']])
                            date_str = date_str.replace('--', '').strip()
                            if date_str:
                                try:
                                    # Try to parse the date string
                                    date_obj = datetime.strptime(date_str, '%d-%m-%Y')
                                    date_values.append(date_obj)
                                except:
                                    try:
                                        date_obj = datetime.strptime(date_str, '%Y-%m-%d')
                                        date_values.append(date_obj)
                                    except:
                                        date_values.append(date_str)
                        is_date_string = True
                    # Handle 1D string arrays
                    elif len(data.shape) == 1:
                        from datetime import datetime
                        for i in range(min(N, len(data))):
                            date_str = data[i].decode('utf-8') if isinstance(data[i], bytes) else str(data[i])
                            date_str = date_str.strip()
                            if date_str:
                                try:
                                    date_obj = datetime.strptime(date_str, '%d-%m-%Y')
                                    date_values.append(date_obj)
                                except:
                                    try:
                                        date_obj = datetime.strptime(date_str, '%Y-%m-%d')
                                        date_values.append(date_obj)
                                    except:
                                        date_values.append(date_str)
                        is_date_string = True
                except Exception as e:
                    print(f"   ⚠️  Could not parse date strings: {e}")
            
            # Check if this is a numeric time variable and convert to datetime
            is_time_var = False
            time_data = None
            if var_name.lower() in ['time', 'date', 'datetime'] and not is_date_string:
                try:
                    # Get time units from attributes
                    if hasattr(var, 'units'):
                        time_units = var.units
                        if hasattr(var, 'calendar'):
                            calendar = var.calendar
                        else:
                            calendar = 'standard'
                        
                        # Convert to datetime using netCDF4's num2date
                        from netCDF4 import num2date
                        time_data = num2date(data, units=time_units, calendar=calendar)
                        is_time_var = True
                        print(f"   Time units: {time_units}")
                        print(f"   Calendar: {calendar}")
                except Exception as e:
                    print(f"   ⚠️  Could not convert to datetime: {e}")
            
            # Display first N values - String-based dates
            if is_date_string and len(date_values) > 0:
                print(f"   First {len(date_values)} dates:")
                
                # Format dates nicely
                date_strings = []
                for date_val in date_values:
                    if hasattr(date_val, 'strftime'):
                        date_strings.append(date_val.strftime('%d-%m-%Y'))
                    else:
                        date_strings.append(str(date_val))
                
                # Print dates in groups of 5 for readability
                for i in range(0, len(date_strings), 5):
                    chunk = date_strings[i:i+5]
                    print(f"   {', '.join(chunk)}")
                
                # Show date range statistics (sample first and last to avoid processing all dates)
                if len(date_values) > 0 and hasattr(date_values[0], 'strftime'):
                    try:
                        print(f"\n   📅 DATE RANGE:")
                        print(f"   {'-'*60}")
                        print(f"   First date : {date_values[0].strftime('%d-%m-%Y')}")
                        
                        # Get the last date by parsing only the last record
                        if len(data.shape) == 2 and data.shape[0] > 0:
                            last_date_str = ''.join([char.decode('utf-8') if isinstance(char, bytes) else str(char) 
                                              for char in data[-1] if char not in [b'', '', b'--', '--']])
                            last_date_str = last_date_str.replace('--', '').strip()
                            if last_date_str:
                                from datetime import datetime
                                try:
                                    last_date = datetime.strptime(last_date_str, '%d-%m-%Y')
                                except:
                                    try:
                                        last_date = datetime.strptime(last_date_str, '%Y-%m-%d')
                                    except:
                                        last_date = date_values[0]
                                print(f"   Last date  : {last_date.strftime('%d-%m-%Y')}")
                            else:
                                print(f"   Last date  : {date_values[-1].strftime('%d-%m-%Y')}")
                        else:
                            print(f"   Last date  : {date_values[-1].strftime('%d-%m-%Y')}")
                        
                        print(f"   Total dates: {data.shape[0] if len(data.shape) == 2 else len(data)}")
                    except Exception as e:
                        print(f"   ⚠️  Could not compute full date range: {e}")
            
            # Display first N values - Numeric time variables
            elif is_time_var and time_data is not None:
                # For time variables, show dates
                n_to_show = min(N, len(time_data.flatten()))
                flat_time = time_data.flatten()
                print(f"   First {n_to_show} dates:")
                
                # Format dates nicely
                date_strings = []
                for i in range(n_to_show):
                    if hasattr(flat_time[i], 'strftime'):
                        date_strings.append(flat_time[i].strftime('%Y-%m-%d %H:%M:%S'))
                    else:
                        date_strings.append(str(flat_time[i]))
                
                # Print dates in groups of 5 for readability
                for i in range(0, len(date_strings), 5):
                    chunk = date_strings[i:i+5]
                    print(f"   {', '.join(chunk)}")
                
                # Show date range statistics
                print(f"\n   📅 DATE RANGE:")
                print(f"   {'-'*60}")
                print(f"   First date : {flat_time[0]}")
                print(f"   Last date  : {flat_time[-1]}")
                print(f"   Total dates: {len(time_data)}")
                
            else:
                # For non-time variables, show numeric values as before
                flat_data = data.flatten()
                n_to_show = min(N, len(flat_data))
                print(f"   First {n_to_show} values (flattened):")
                
                # Format output based on data type
                if np.issubdtype(data.dtype, np.floating):
                    # For floating point, show with appropriate precision
                    values_str = ", ".join([f"{val:.6f}" for val in flat_data[:n_to_show]])
                elif np.issubdtype(data.dtype, np.integer):
                    # For integers
                    values_str = ", ".join([str(val) for val in flat_data[:n_to_show]])
                else:
                    # For other types (strings, etc.)
                    values_str = ", ".join([str(val) for val in flat_data[:n_to_show]])
                
                # Wrap long lines
                if len(values_str) > 65:
                    print(f"   [{values_str[:65]}...")
                    print(f"    ...{values_str[65:]}]")
                else:
                    print(f"   [{values_str}]")
            
            # COMPREHENSIVE STATISTICS (like pandas describe())
            # Only show numeric statistics for non-time variables
            if np.issubdtype(data.dtype, np.number) and not is_time_var:
                print(f"\n   📊 STATISTICS (like pd.DataFrame.describe()):")
                print(f"   {'-'*60}")
                
                # Remove masked/fill values and NaN
                if hasattr(data, 'mask'):
                    valid_data = data[~data.mask]
                else:
                    valid_data = data[~np.isnan(data)] if np.issubdtype(data.dtype, np.floating) else data
                
                if len(valid_data) > 0:
                    # Calculate all statistics
                    count = len(valid_data)
                    mean_val = np.mean(valid_data)
                    std_val = np.std(valid_data, ddof=1) if count > 1 else 0.0
                    min_val = np.min(valid_data)
                    q25 = np.percentile(valid_data, 25)
                    q50 = np.percentile(valid_data, 50)  # median
                    q75 = np.percentile(valid_data, 75)
                    max_val = np.max(valid_data)
                    
                    # Additional useful statistics
                    nan_count = np.sum(np.isnan(data)) if np.issubdtype(data.dtype, np.floating) else 0
                    total_count = data.size
                    
                    # Print in a formatted table
                    print(f"   {'Statistic':<15} {'Value':>15}")
                    print(f"   {'-'*31}")
                    print(f"   {'count':<15} {count:>15,}")
                    print(f"   {'total (w/NaN)':<15} {total_count:>15,}")
                    print(f"   {'NaN count':<15} {nan_count:>15,}")
                    print(f"   {'mean':<15} {mean_val:>15.6f}")
                    print(f"   {'std':<15} {std_val:>15.6f}")
                    print(f"   {'min':<15} {min_val:>15.6f}")
                    print(f"   {'25%':<15} {q25:>15.6f}")
                    print(f"   {'50% (median)':<15} {q50:>15.6f}")
                    print(f"   {'75%':<15} {q75:>15.6f}")
                    print(f"   {'max':<15} {max_val:>15.6f}")
                    
                    # Additional statistics
                    range_val = max_val - min_val
                    iqr = q75 - q25
                    print(f"   {'-'*31}")
                    print(f"   {'range':<15} {range_val:>15.6f}")
                    print(f"   {'IQR (Q3-Q1)':<15} {iqr:>15.6f}")
                    
                    # Skewness and Kurtosis (if scipy available)
                    try:
                        from scipy import stats
                        skew = stats.skew(valid_data)
                        kurt = stats.kurtosis(valid_data)
                        print(f"   {'skewness':<15} {skew:>15.6f}")
                        print(f"   {'kurtosis':<15} {kurt:>15.6f}")
                    except ImportError:
                        pass
                else:
                    print(f"   ⚠️  No valid numeric data available")
            
        except Exception as e:
            print(f"   ✗ Error reading data: {str(e)}")
        
        print()
    
    # Close the dataset
    dataset.close()
    print("✓ Dataset closed")
else:
    print("⚠ Dataset not opened. Please check the file path.")

VARIABLE STATISTICS & FIRST 10 VALUES

📌 date
   Shape: (4018, 10), Dimensions: ('time', 'date_strlen')
   First 10 dates:
   01-10-2025, 02-10-2025, 03-10-2025, 04-10-2025, 05-10-2025
   06-10-2025, 07-10-2025, 08-10-2025, 09-10-2025, 10-10-2025

   📅 DATE RANGE:
   ------------------------------------------------------------
   First date : 01-10-2025
   Last date  : 30-09-2025
   Total dates: 4018

📌 lat
   Shape: (12,), Dimensions: ('lat',)
   First 10 values (flattened):
   [29.950001, 30.049999, 30.150000, 30.250000, 30.350000, 30.450001,...
    ... 30.549999, 30.650000, 30.750000, 30.850000]

   📊 STATISTICS (like pd.DataFrame.describe()):
   ------------------------------------------------------------
   Statistic                 Value
   -------------------------------
   count                         1
   total (w/NaN)                12
   NaN count                     0
   mean                  30.500000
   std                    0.000000
   min                   29.950001
 

C:\Users\AlienX\AppData\Roaming\Python\Python313\site-packages\numpy\lib\_function_base_impl.py:4809: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(


   ✗ Error reading data: unsupported format string passed to numpy.ndarray.__format__

📌 lon
   Shape: (12,), Dimensions: ('lon',)
   First 10 values (flattened):
   [74.949997, 75.050003, 75.150002, 75.250000, 75.349998, 75.449997,...
    ... 75.550003, 75.650002, 75.750000, 75.849998]

   📊 STATISTICS (like pd.DataFrame.describe()):
   ------------------------------------------------------------
   Statistic                 Value
   -------------------------------
   count                         1
   total (w/NaN)                12
   NaN count                     0
   mean                  75.500000
   std                    0.000000
   min                   74.949997
   25%                   75.224998
   50% (median)          75.500000
   75%                   75.775002
   max                   76.050003
   -------------------------------
   range                  1.100006
   IQR (Q3-Q1)            0.550003
   ✗ Error reading data: unsupported format string passed to numpy.ndarray

C:\Users\AlienX\AppData\Roaming\Python\Python313\site-packages\numpy\lib\_function_base_impl.py:4809: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(


## Method 2: Using xarray (if available)

xarray provides a more convenient interface for working with NetCDF files.

In [6]:
# if XARRAY_AVAILABLE:
#     print("="*70)
#     print("READING WITH XARRAY")
#     print("="*70)
#     print()
    
#     try:
#         # Open dataset with xarray
#         ds = xr.open_dataset(nc_file_path)
        
#         # Display dataset summary
#         print("📊 DATASET SUMMARY:")
#         print("-" * 70)
#         print(ds)
#         print()
        
#         # Display first N values for each data variable
#         print(f"📌 FIRST {N} VALUES (using xarray):")
#         print("-" * 70)
        
#         for var_name in ds.data_vars:
#             print(f"\n{var_name}:")
#             var_data = ds[var_name]
            
#             # Get flattened values
#             flat_values = var_data.values.flatten()[:N]
            
#             print(f"  Shape: {var_data.shape}")
#             print(f"  Dimensions: {var_data.dims}")
#             print(f"  First {len(flat_values)} values:")
#             print(f"  {flat_values}")
            
#             # Show attributes
#             if var_data.attrs:
#                 print(f"  Attributes: {dict(var_data.attrs)}")
        
#         # Close dataset
#         ds.close()
#         print("\n✓ Dataset closed")
        
#     except Exception as e:
#         print(f"✗ Error reading with xarray: {str(e)}")
# else:
#     print("⚠ xarray not available. Skipping xarray method.")
#     print("  Install with: pip install xarray")

## Custom Reader Function

A reusable function to read any NetCDF file:

In [7]:
def read_nc_file_summary(file_path, n_values=10, show_stats=True):
    """
    Read and display summary of a NetCDF file with comprehensive statistics.
    
    Parameters:
    -----------
    file_path : str
        Path to the NetCDF file
    n_values : int
        Number of values to display from each variable (default: 10)
    show_stats : bool
        Whether to show statistics for numeric variables (default: True)
    
    Returns:
    --------
    dict : Dictionary containing file information and statistics
    """
    
    if not os.path.exists(file_path):
        print(f"✗ File not found: {file_path}")
        return None
    
    print(f"\n{'='*70}")
    print(f"READING: {file_path}")
    print(f"{'='*70}\n")
    
    try:
        dataset = nc.Dataset(file_path, 'r')
        
        file_info = {
            'dimensions': {},
            'variables': {},
            'attributes': {}
        }
        
        # Global attributes
        print("📋 Global Attributes:")
        for attr in dataset.ncattrs():
            file_info['attributes'][attr] = dataset.getncattr(attr)
            print(f"  {attr}: {dataset.getncattr(attr)}")
        print()
        
        # Dimensions
        print("📏 Dimensions:")
        for dim_name, dim in dataset.dimensions.items():
            file_info['dimensions'][dim_name] = len(dim)
            print(f"  {dim_name}: {len(dim)}")
        print()
        
        # Variables with comprehensive statistics
        print(f"📊 Variables & Statistics:\n")
        for var_name, var in dataset.variables.items():
            print(f"  {var_name}:")
            print(f"    Shape: {var.shape}")
            print(f"    Type: {var.dtype}")
            
            # Get data
            data = var[:]
            
            # Check if this is a Date/time variable stored as strings
            is_date_string = False
            date_values = []
            if var_name.lower() in ['date', 'datetime'] and data.dtype.kind in ['S', 'U', 'O']:
                try:
                    from datetime import datetime
                    # Handle character array dates (e.g., shape (n, strlen))
                    if len(data.shape) == 2:
                        for i in range(min(n_values, data.shape[0])):
                            date_str = ''.join([char.decode('utf-8') if isinstance(char, bytes) else str(char) 
                                              for char in data[i] if char not in [b'', '', b'--', '--']])
                            date_str = date_str.replace('--', '').strip()
                            if date_str:
                                try:
                                    date_obj = datetime.strptime(date_str, '%d-%m-%Y')
                                    date_values.append(date_obj.strftime('%d-%m-%Y'))
                                except:
                                    try:
                                        date_obj = datetime.strptime(date_str, '%Y-%m-%d')
                                        date_values.append(date_obj.strftime('%Y-%m-%d'))
                                    except:
                                        date_values.append(date_str)
                        is_date_string = True
                    # Handle 1D string arrays
                    elif len(data.shape) == 1:
                        for i in range(min(n_values, len(data))):
                            date_str = data[i].decode('utf-8') if isinstance(data[i], bytes) else str(data[i])
                            date_str = date_str.strip()
                            if date_str:
                                try:
                                    date_obj = datetime.strptime(date_str, '%d-%m-%Y')
                                    date_values.append(date_obj.strftime('%d-%m-%Y'))
                                except:
                                    try:
                                        date_obj = datetime.strptime(date_str, '%Y-%m-%d')
                                        date_values.append(date_obj.strftime('%Y-%m-%d'))
                                    except:
                                        date_values.append(date_str)
                        is_date_string = True
                except Exception as e:
                    print(f"    ⚠️  Could not parse date strings: {e}")
            
            if is_date_string and len(date_values) > 0:
                print(f"    First {len(date_values)} dates: {date_values}")
            else:
                flat_data = data.flatten()
                n_show = min(n_values, len(flat_data))
                print(f"    First {n_show}: {flat_data[:n_show]}")
            
            # Comprehensive statistics (like pandas describe())
            if show_stats and np.issubdtype(data.dtype, np.number):
                print(f"\n    📊 Statistics (describe-like):")
                
                # Remove masked/fill values and NaN
                if hasattr(data, 'mask'):
                    valid_data = data[~data.mask]
                else:
                    valid_data = data[~np.isnan(data)] if np.issubdtype(data.dtype, np.floating) else data
                
                if len(valid_data) > 0:
                    stats_dict = {}
                    
                    # Calculate statistics
                    count = len(valid_data)
                    mean_val = float(np.mean(valid_data))
                    std_val = float(np.std(valid_data, ddof=1)) if count > 1 else 0.0
                    min_val = float(np.min(valid_data))
                    q25 = float(np.percentile(valid_data, 25))
                    q50 = float(np.percentile(valid_data, 50))
                    q75 = float(np.percentile(valid_data, 75))
                    max_val = float(np.max(valid_data))
                    
                    nan_count = int(np.sum(np.isnan(data))) if np.issubdtype(data.dtype, np.floating) else 0
                    total_count = int(data.size)
                    
                    # Store in dictionary
                    stats_dict = {
                        'count': count,
                        'total': total_count,
                        'nan_count': nan_count,
                        'mean': mean_val,
                        'std': std_val,
                        'min': min_val,
                        '25%': q25,
                        '50%': q50,
                        '75%': q75,
                        'max': max_val,
                        'range': max_val - min_val,
                        'IQR': q75 - q25
                    }
                    
                    # Print formatted
                    print(f"       count       : {count:,}")
                    print(f"       mean        : {mean_val:.6f}")
                    print(f"       std         : {std_val:.6f}")
                    print(f"       min         : {min_val:.6f}")
                    print(f"       25%         : {q25:.6f}")
                    print(f"       50% (median): {q50:.6f}")
                    print(f"       75%         : {q75:.6f}")
                    print(f"       max         : {max_val:.6f}")
                    
                    # Add to file_info
                    file_info['variables'][var_name] = {
                        'shape': var.shape,
                        'dtype': str(var.dtype),
                        'first_values': flat_data[:n_show].tolist(),
                        'statistics': stats_dict
                    }
                else:
                    print(f"       No valid numeric data")
                    file_info['variables'][var_name] = {
                        'shape': var.shape,
                        'dtype': str(var.dtype),
                        'first_values': flat_data[:n_show].tolist()
                    }
            else:
                file_info['variables'][var_name] = {
                    'shape': var.shape,
                    'dtype': str(var.dtype),
                    'first_values': flat_data[:n_show].tolist()
                }
            
            print()
        
        dataset.close()
        print("✓ File read successfully\n")
        
        return file_info
        
    except Exception as e:
        print(f"✗ Error: {str(e)}")
        return None

print("✓ Function 'read_nc_file_summary' defined")
print("  Usage: read_nc_file_summary('path/to/file.nc', n_values=10)")


✓ Function 'read_nc_file_summary' defined
  Usage: read_nc_file_summary('path/to/file.nc', n_values=10)


## Example Usage

Use the function to read your NetCDF file:

## Summary Statistics Table

Create a pandas-like describe() table for all numeric variables:

In [8]:
def create_statistics_table(nc_file_path):
    """
    Create a pandas DataFrame with describe()-like statistics for all numeric variables.
    
    Parameters:
        nc_file_path (str): Path to the NetCDF file
        
    Returns:
        pd.DataFrame: Statistics table with variables as columns and stats as rows
    """
    import pandas as pd
    import numpy as np
    import netCDF4 as nc
    
    try:
        from scipy.stats import skew, kurtosis
        has_scipy = True
    except ImportError:
        has_scipy = False
    
    # Open the NetCDF file
    ds = nc.Dataset(nc_file_path, 'r')
    
    # Dictionary to store statistics for each variable
    stats_dict = {}
    
    # Get all numeric variables (excluding coordinate variables)
    coord_vars = {'lat', 'latitude', 'lon', 'longitude', 'time', 'x', 'y'}
    
    for var_name in ds.variables:
        if var_name.lower() in coord_vars:
            continue
            
        var = ds.variables[var_name]
        
        # Skip non-numeric variables
        if var.dtype.kind not in ['f', 'i', 'u']:
            continue
        
        # Read data
        data = var[:]
        
        # Handle masked arrays
        if isinstance(data, np.ma.MaskedArray):
            valid_data = data.compressed()
        else:
            valid_data = data[~np.isnan(data)]
        
        if len(valid_data) == 0:
            continue
        
        # Calculate statistics
        stats = {
            'count': len(valid_data),
            'mean': np.mean(valid_data),
            'std': np.std(valid_data, ddof=1),  # Sample std (ddof=1)
            'min': np.min(valid_data),
            '25%': np.percentile(valid_data, 25),
            '50%': np.percentile(valid_data, 50),
            '75%': np.percentile(valid_data, 75),
            'max': np.max(valid_data)
        }
        
        # Add skewness and kurtosis if scipy is available
        if has_scipy:
            stats['skew'] = skew(valid_data)
            stats['kurtosis'] = kurtosis(valid_data)
        
        stats_dict[var_name] = stats
    
    ds.close()
    
    # Create DataFrame
    df = pd.DataFrame(stats_dict)
    
    return df

# Example: Create statistics table for a file
file_path = 'data_nc/IMERG_precip/IMERG_Final_30N-31N_75E-76E_2017-2021.nc'
if os.path.exists(file_path):
    stats_table = create_statistics_table(file_path)
    print("\n=== Summary Statistics (pandas describe-like) ===")
    print(stats_table)
else:
    print(f"File not found: {file_path}")

File not found: data_nc/IMERG_precip/IMERG_Final_30N-31N_75E-76E_2017-2021.nc


In [9]:
# Example: Read a NetCDF file
file_info = read_nc_file_summary(nc_file_path, n_values=N)

# The function returns a dictionary with all the information
if file_info:
    print("\n" + "="*70)
    print("FILE INFO DICTIONARY:")
    print("="*70)
    print(f"Number of dimensions: {len(file_info['dimensions'])}")
    print(f"Number of variables: {len(file_info['variables'])}")
    print(f"Number of global attributes: {len(file_info['attributes'])}")


READING: G:\SM2RAIN-irrigation_Final\GPM_IMERG_ludhiana_final_run.nc

📋 Global Attributes:
  title: GPM IMERG Daily Precipitation - ROI Filtered
  institution: NASA Goddard Space Flight Center
  source: GPM_3IMERGDF v07
  history: Created on 2026-02-07 01:39:42
  references: https://doi.org/10.5067/GPM/IMERGDF/DAY/07
  comment: Filtered for ROI: lon(74.95-76.05), lat(29.95-31.05)
  Conventions: CF-1.6

📏 Dimensions:
  time: 4018
  date_strlen: 10
  lat: 12
  lon: 12

📊 Variables & Statistics:

  date:
    Shape: (4018, 10)
    Type: |S1
    First 10 dates: ['01-10-2025', '02-10-2025', '03-10-2025', '04-10-2025', '05-10-2025', '06-10-2025', '07-10-2025', '08-10-2025', '09-10-2025', '10-10-2025']
✗ Error: cannot access local variable 'flat_data' where it is not associated with a value


## Batch Processing

Read multiple NetCDF files from a directory:

In [10]:
def read_multiple_nc_files(directory, pattern='*.nc', n_values=5):
    """
    Read multiple NetCDF files from a directory.
    
    Parameters:
    -----------
    directory : str
        Path to directory containing NetCDF files
    pattern : str
        File pattern to match (default: '*.nc')
    n_values : int
        Number of values to display per variable (default: 5)
    """
    from glob import glob
    
    nc_files = glob(os.path.join(directory, pattern))
    
    if not nc_files:
        print(f"✗ No NetCDF files found in {directory} matching pattern '{pattern}'")
        return
    
    print(f"Found {len(nc_files)} NetCDF files\n")
    
    for i, file_path in enumerate(nc_files, 1):
        print(f"\n{'#'*70}")
        print(f"FILE {i}/{len(nc_files)}: {os.path.basename(file_path)}")
        print(f"{'#'*70}")
        read_nc_file_summary(file_path, n_values=n_values, show_stats=False)

print("✓ Function 'read_multiple_nc_files' defined")
print("  Usage: read_multiple_nc_files('data_nc/ERA5_LST/', pattern='*.nc', n_values=5)")

✓ Function 'read_multiple_nc_files' defined
  Usage: read_multiple_nc_files('data_nc/ERA5_LST/', pattern='*.nc', n_values=5)


In [11]:
# Example: Read all NetCDF files in a directory
# Uncomment and modify the line below to use:

# read_multiple_nc_files('data_nc/ERA5_LST/', pattern='*.nc', n_values=5)

print("💡 TIP: Uncomment the line above and modify the directory path to read multiple files")

💡 TIP: Uncomment the line above and modify the directory path to read multiple files


## Summary

This notebook provides three ways to read NetCDF files:

1. **Direct method**: Using `netCDF4` library for detailed access
2. **xarray method**: More convenient interface (if xarray is installed)
3. **Function method**: Reusable function for quick inspection

### Quick Tips:
- Change `nc_file_path` to point to your NetCDF file
- Adjust `N` to show more or fewer values
- Use `read_nc_file_summary()` for quick file inspection
- Use `read_multiple_nc_files()` to process multiple files at once